# AutoLineage quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kishanraj41/autolineage/blob/main/examples/quickstart.ipynb)

Zero-code data lineage for Python ML pipelines. One import tracks every pandas and scikit-learn operation, then catches anomalies and points at the operation that broke a metric.

Run this top to bottom (Runtime > Run all). About one minute, no data download, no API keys.

1. Install and activate tracking with one import
2. Run a normal pipeline and inspect the captured lineage
3. Plant a one-line bug, let AutoLineage catch it and name the operation responsible

Repo: https://github.com/kishanraj41/autolineage  |  PyPI: https://pypi.org/project/autolineage/  |  Paper: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=6683825

In [1]:
%pip install -q "autolineage[sklearn]"

## 1. Activate tracking with one import

`import autolineage.auto` patches pandas and scikit-learn at import time. Put it before you import the framework symbols you want tracked: a `from sklearn.metrics import f1_score` that runs *before* this line binds to the original function and bypasses the hook (AutoLineage warns you when that happens).

In [2]:
import autolineage.auto  # the entire setup

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

AutoLineage: 239 hooks installed across 3 frameworks (pandas-io, pandas-transforms, sklearn)


## 2. Run a normal pipeline

A small synthetic fraud dataset so the notebook is self-contained: 20,000 transactions, 2% fraud, and fraud skews toward high amounts. Use pandas and scikit-learn exactly as you always would. AutoLineage records each operation in the background.

In [3]:
def make_transactions(n=20_000, seed=42):
    rng = np.random.default_rng(seed)
    n_fraud = int(n * 0.02)
    n_ok = n - n_fraud
    ok = pd.DataFrame({
        "amount": rng.gamma(2.0, 30.0, n_ok),
        "v1": rng.normal(0, 1, n_ok), "v2": rng.normal(0, 1, n_ok), "v3": rng.normal(0, 1, n_ok),
        "fraud": 0,
    })
    fraud = pd.DataFrame({
        "amount": rng.gamma(2.0, 90.0, n_fraud),
        "v1": rng.normal(-2, 1.5, n_fraud), "v2": rng.normal(2.5, 1.5, n_fraud), "v3": rng.normal(-1.5, 1, n_fraud),
        "fraud": 1,
    })
    return pd.concat([ok, fraud], ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)

df = make_transactions().dropna().drop_duplicates()
df = df[df["amount"] < df["amount"].quantile(0.999)]   # trim extreme outliers

X, y = df.drop(columns=["fraud"]), df["fraud"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
model = RandomForestClassifier(n_estimators=50, random_state=0).fit(X_tr, y_tr)
score = f1_score(y_te, model.predict(X_te))
print(f"rows after filter: {len(df):,}   F1 = {score:.4f}")

rows after filter: 19,980   F1 = 0.8027


### Inspect the captured lineage

Every operation above, from `dropna` through `f1_score`, is now in one graph. The tracker prints a summary, exports Mermaid and Graphviz DOT, and renders an interactive HTML graph inline.

In [4]:
from autolineage.auto import get_tracker

tracker = get_tracker()
print(tracker)
print()
print(tracker.to_mermaid())   # paste into any Mermaid viewer, or a GitHub markdown file

<UnifiedTracker: 11 operations across pandas-transforms, sklearn>

graph TD
  n0["1. concat<br/>(20000, 5) · 2ms"]
  n1["2. sample<br/>(20000, 5) · 6ms"]
  n2["3. reset_index<br/>(20000, 5) · 0ms"]
  n3["4. dropna<br/>(20000, 5) · 1ms"]
  n4["5. drop_duplicates<br/>(20000, 5) · 4ms"]
  n5["6. filter<br/>(20000, 5) → (19980, 5) · -20 rows"]
  n6["7. drop<br/>(19980, 5) → (19980, 4) · 1ms"]
  n7["8. train_test_split<br/>(19980, 4) · -3,996 rows · 27ms"]
  n8["9. RandomForestClassifier.fit<br/>(15984, 4) · 1679ms"]
  n9["10. RandomForestClassifier.predict<br/>(3996, 4) → (3996,) · 22ms"]
  n10["11. f1_score<br/>= 0.8027 · 5ms"]
  n0 --> n1
  n1 --> n2
  n2 --> n3
  n3 --> n4
  n4 --> n5
  n5 --> n6
  n6 --> n7
  n6 --> n7
  n7 --> n8
  n7 --> n8
  n7 --> n9
  n8 --> n9
  n9 --> n10
  classDef io fill:#fff4cc,stroke:#caa400,color:#222;
  classDef transform fill:#cfe2ff,stroke:#3a76d8,color:#222;
  classDef split fill:#ffd9b3,stroke:#cc6f00,color:#222;
  classDef preprocess fill:#e6d6ff,str

In [5]:
from autolineage.viz import to_jupyter

to_jupyter(tracker)   # interactive graph; click a node for shapes, timings and upstream operations

## 3. Catch a broken pipeline automatically

This is what AutoLineage is for. The function below is the same pipeline, with one line that differs: the `buggy` version filters on the 5th percentile of `amount` instead of the 99.9th. Nothing crashes. The model still trains and reports a number.

Each run gets a fresh tracker (`stop_tracking()` then `start_tracking()`), and the scikit-learn imports happen after tracking starts so they bind to the hooked versions.

In [6]:
from autolineage.auto import start_tracking, stop_tracking
from autolineage.core.analyzer import LineageAnalyzer

def run_pipeline(buggy=False):
    stop_tracking()
    tracker = start_tracking()
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import f1_score

    df = make_transactions().dropna().drop_duplicates()
    q = 0.05 if buggy else 0.999          # <-- the only line that differs
    df = df[df["amount"] < df["amount"].quantile(q)]

    X, y = df.drop(columns=["fraud"]), df["fraud"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
    model = RandomForestClassifier(n_estimators=50, random_state=0).fit(X_tr, y_tr)
    f1 = f1_score(y_te, model.predict(X_te), zero_division=0)
    print(f"[{'buggy' if buggy else 'clean'}] rows after filter: {len(df):,}   F1 = {f1:.4f}")
    return tracker

# Healthy run: save its fingerprint as the baseline
clean_tracker = run_pipeline(buggy=False)
LineageAnalyzer(clean_tracker).save_fingerprint("baseline.json", append=False)

# Someone changes the filter. Compare against the baseline.
buggy_tracker = run_pipeline(buggy=True)
analyzer = LineageAnalyzer(buggy_tracker)
analyzer.load_baseline("baseline.json")

print()
for a in analyzer.detect_anomalies():
    print(f"[{a.severity}] {a.message}")

print()
cause = analyzer.localize_root_cause("f1_score")
print(cause.explanation)

AutoLineage: 239 hooks installed across 3 frameworks (pandas-io, pandas-transforms, sklearn)


[clean] rows after filter: 19,980   F1 = 0.8027
AutoLineage: 239 hooks installed across 3 frameworks (pandas-io, pandas-transforms, sklearn)
[buggy] rows after filter: 1,000   F1 = 0.0000

[critical] filter row change: -19,000 (baseline: -20, 94900% deviation)
[critical] train_test_split row change: -200 (baseline: -3,996, 95% deviation)
[critical] f1_score dropped from 0.8027 to 0.0000 (-100.0%)

The most likely cause of f1_score degradation (from 0.8027 to 0.0000) is 'filter' at step 5. Row change was -19,000 (baseline: -20). 


The analyzer flagged the row-count collapse at `filter`, the F1 collapse at `f1_score`, and named `filter` as the root cause. No decorators, no logging calls, no changes to the pipeline.

## Next steps

- `pip install autolineage[all]` adds PySpark hooks and the Jupyter extras.
- `python examples/anomaly_demo.py` in the repo runs the same demo on a larger pipeline with `StandardScaler` and four metrics.
- `tracker.visualize("trace.html")` writes the interactive graph to disk; `tracker.to_dot()` gives Graphviz.
- Open an issue or star the repo: https://github.com/kishanraj41/autolineage